In [3]:
### takes a dataframe row and returns an origin annotation 
def tax_label(row):
   
    x=row['lineage']
    if 'Fungi' in x:
        return 'Fungi'
    elif 'Viruses' in x:
        return 'Viruses'
    elif 'Bacteria' in x:
        return 'Bacteria'
    elif 'Viridiplantae' in x:
        return 'Viridiplantae'
    elif 'primary_chimera' in str(row):
        return 'chimera'
    elif row['secondary']=='True':
        return 'chimera'
    elif 'Arthropoda' in x:
        return 'Arthropoda'
    elif 'Metazoa' in x:
        return 'Metazoa' 
    elif 'Sar' in x:
        return 'Sar'
    elif 'Evosea' in x:
        return 'Evosea'
    elif 'Metamonada' in x:
        return x
    elif 'Archaea' in x:
        return 'Archaea'
    else:
        return x
        

In [4]:
### takes a dataframe row and returns an origin annotation 
def tax_label2(row):
   
    x=row['lineage']
    if 'Fungi' in x:
        return 'Fungi'
    elif 'Viruses' in x:
        return 'Viruses'
    elif 'Bacteria' in x:
        return 'Bacteria'
    elif 'Viridiplantae' in x:
        return 'Viridiplantae'
    elif 'primary_chimera' in str(row):
        return 'Arthropoda'
    elif row['secondary']=='True':
        return 'Arthropoda'
    elif 'Arthropoda' in x:
        return 'Arthropoda'
    elif 'Metazoa' in x:
        return 'Metazoa' 
    elif 'Sar' in x:
        return 'Sar'
    elif 'Evosea' in x:
        return 'Evosea'
    elif 'Metamonada' in x:
        return x
    elif 'Archaea' in x:
        return 'Archaea'
    else:
        return x
        

In [5]:
## returns lowest spanning taxonomic rank and its taxid for a list of taxids
def get_common_ancestor(taxids):
    tree = ncbi.get_topology(taxids)
    
    taxid=tree.taxid
    return tree.sci_name, ncbi.get_rank([taxid])[taxid]

In [6]:
from ete3 import NCBITaxa
ncbi = NCBITaxa()

In [7]:
## for category II and IV hgt topologies, find broadest monophyletic group of arthropod sequences containing HGT-chimeras
## in this clade, lookup common ancestor of chimeric and non-chimeric sequences
## exclude cases in which closest clade contains both arthropod and non-metazoan sequences
## exclude cases in which all hgt-chimeras are not in the same monophyletic group
for x in idf[idf.topological_category.isin(['II','IV'])&~idf.tree_Note.astype(str).str.contains("Closest clade contains both non-chimeric arthropod and non-metazoan sequences.")].index:
    interv=x.replace(" ","")
    try:
       
        # try:
        t=Tree(f'outputs/phylogenetic_data_filtered/{interv}/final_rooted.tree')
        combined=pd.read_csv(f'outputs/phylogenetic_data_filtered/{interv}/combined_sequences_data.tsv',sep='\t')
        try:
            combined=combined.set_index('target_name')
        except:
            combined=combined.set_index('sseqid')
        
        tax_map={}
        for index, row in combined.iterrows():
            if ';' in index:
                tax_map[index.split(";")[1]]=tax_label(row)
            else:
                tax_map[index]=tax_label(row)
                
        tax_map2={}
        for index, row in combined.iterrows():
            if ';' in index:
                tax_map2[index.split(";")[1]]=tax_label2(row)
            else:
                tax_map2[index]=tax_label2(row)
        
        for node in t:
            node.add_features(taxname=tax_map[node.name])
        
        for node in t:
            node.add_features(taxname2=tax_map2[node.name])
        for node in t:
            tid=combined[combined.index.str.contains(node.name)].taxid.values[0].astype(int)
            node.add_features(taxid=tid)
        
        n_chimeras_clades=len([node for node in t.get_monophyletic(values=["chimera"], target_attr="taxname")])
        chimeras=set([x.name for x in t if x.taxname=='chimera'])
        ## exclude cases in which all hgt-chimeras are not in the same monophyletic group
        if n_chimeras_clades==1: 
            for node in t.get_monophyletic(values=["Arthropoda"], target_attr="taxname2"):
                ## if chimeras are in the clade
                if len(chimeras-set([x.name for x in node]))==0:
                    chimera_taxids=[x.taxid for x in node if x.name in chimeras]
                    non_chimera_taxids=[x.taxid for x in node if x.name not in chimeras]
                    idf.loc[x,['chimera_MRCA_name','chimera_MRCA_rank']]=get_common_ancestor(chimera_taxids)
                    idf.loc[x,['non_chimera_MRCA_name','non_chimera_MRCA_rank']]=get_common_ancestor(non_chimera_taxids)
                    break
        else:
            idf.loc[x,'chimera_monophyletic']=False
    except:
        print(interv)


GCF_025399875.1;XP_065173386.1;HGT_(788,1288)


In [12]:
##Manually fill in a failed lookup
interv='GCF_025399875.1; XP_065173386.1; HGT_(788,1288)'
idf.loc[interv,['non_chimera_MRCA_name','non_chimera_MRCA_rank']]=['Polyphaga', 'suborder']

idf.loc[interv,['chimera_MRCA_name','chimera_MRCA_rank']]=['Dalotia coriaria', 'species']

In [27]:
for index, row in idf[idf.chimera_MRCA_name.astype(str)!='nan'].iterrows():
    non_chimera_MRCA=row.non_chimera_MRCA_name
    chimera_MRCA=row.chimera_MRCA_name
    non_chimera_tid=ncbi.get_name_translator([non_chimera_MRCA])[non_chimera_MRCA][0]
    chimera_tid=ncbi.get_name_translator([chimera_MRCA])[chimera_MRCA][0]
    idf.loc[index,'chimera_younger']= chimera_tid!=non_chimera_tid and non_chimera_tid in ncbi.get_lineage(chimera_tid)

In [35]:
idf[idf.chimera_MRCA_name.astype(str)!='nan'].loc[:,['cluster_id', 'topological_category',
       'chimera_MRCA_name', 'chimera_MRCA_rank', 'non_chimera_MRCA_name',
       'non_chimera_MRCA_rank', 'chimera_younger']].to_csv("SI tables/unfiltered_revision_HGTc_SI table 10_v5.csv")
    